# 03 — Impact analysis

Impact answers: *if I change this column, what else is affected?* (downstream)
or *where does this column come from?* (upstream).

This notebook compares directions, inspects warnings, and shows the exact JSON
the CLI emits via `emit_impact`.

In [ ]:
import json
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _paths import wedge_project

PROJECT_DIR = wedge_project()
SELECTION = "orders.amount"

In [ ]:
from clearmetric.compiler.impact import impact
from clearmetric.emitters.impact import emit_impact

compiled_up, upstream = impact(
    PROJECT_DIR,
    selection=SELECTION,
    direction="upstream",
)
compiled_down, downstream = impact(
    PROJECT_DIR,
    selection=SELECTION,
    direction="downstream",
)

print("upstream related:", len(upstream.related_ids))
print("downstream related:", len(downstream.related_ids))

In [ ]:
print("=== upstream ===")
for node_id in upstream.related_ids:
    print(f"  {node_id}")

print("\n=== downstream ===")
for node_id in downstream.related_ids:
    print(f"  {node_id}")

## Warnings and derivation

Warnings are first-class — unresolved lineage is surfaced loudly, not silently omitted.

In [ ]:
for warning in upstream.warnings:
    print(f"[{warning.code}] subject={warning.subject_id}")
    print(f"  {warning.message}")

impact_json = emit_impact(
    compiled_up,
    upstream,
    format="json",
    direction="upstream",
)
payload = json.loads(impact_json)
print("\nemit_impact keys:", sorted(payload.keys()))
print(f"derivation entries: {len(payload['derivation'])}")

## Validate against `impact-output.schema.json`

Consumer bundles and corpus checks use this exact emitted shape.

In [ ]:
from clearmetric.core.validate import validate_impact_output_dict

validate_impact_output_dict(payload)
print("impact output valid")

## Text and Mermaid renderers

Same traversal result — different presentation for terminals or docs.

In [ ]:
tree = emit_impact(
    compiled_up,
    upstream,
    format="tree",
    direction="upstream",
)
print(tree[:800] + ("..." if len(tree) > 800 else ""))

mermaid = emit_impact(
    compiled_up,
    upstream,
    format="mermaid",
    direction="upstream",
)
print("\n--- mermaid (first lines) ---")
print("\n".join(mermaid.splitlines()[:12]))